# 03 - Cleaning UN DESA Destination Stock Data

This notebook cleans and standardizes UN DESA migration stock data for migrants originating from Cameroon.

Analytical role:

- supports Q1 on destination countries
- keeps migrant stock indicators separate from flow indicators
- prepares a standardized file used later to build `destinations_master.csv`

The cleaning process focuses on keeping the dataset reproducible, simple to inspect and compatible with the project-wide analytical schema.


## 1. Analytical Role in the Project

This dataset supports the first research question of the project:

**Q1: How did the main destination countries of Cameroonian migrants evolve before, during, and after the Covid-19 pandemic?**

UN DESA is used because it provides a global view of migrant stock by destination country.

In this project, migrant stock refers to the number of Cameroonian migrants present in a destination country at a given reference year. It does not measure yearly migration flows or new arrivals.


## 2. Methodological Note: Migrant Stock vs Migration Flow

This dataset measures **migrant stock**, not **migration flow**.

A migrant stock represents the number of Cameroonian migrants present in a destination country at a specific reference year.

It does not indicate how many Cameroonians entered that country during the same year.

For example, if a country has a high Cameroonian migrant stock in 2024, it means many Cameroonian migrants are present there in 2024. However, it does not necessarily mean that this country received the highest number of new Cameroonian migrants in 2024.

This distinction is important to avoid overinterpreting the results.


## 3. Setup and Project Paths

This section imports the required Python modules and defines the project paths used to load raw data and export cleaned data.

The raw dataset is kept in `data/raw/`, while the cleaned version is exported to `data/processed/cleaned/`.


In [20]:
import pandas as pd
from pathlib import Path

In [21]:
PROJECT_ROOT = Path("..")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

GLOBAL_PATH = DATA_RAW / "global"
CLEANED_PATH = DATA_PROCESSED / "cleaned"

CLEANED_PATH.mkdir(parents=True, exist_ok=True)

## 4. Data Loading

The UN DESA raw dataset is loaded from the `data/raw/global/` folder.

Input file used in this notebook:

```text
data/raw/global/undesa_cameroon_global_destinations.csv
```

At this stage, the dataset is loaded without modifying the raw file.


In [22]:
undesa = pd.read_csv(GLOBAL_PATH / "undesa_cameroon_global_destinations.csv")
undesa

,Destination,Location code of destination,Origin,Location code of origin,Year,Total,Male,Female
0,World,900,World,900,1990,153 916 063,77 772 082,76 143 981
1,World,900,Sub-Saharan Africa,1834,1990,14 124 662,7 458 958,6 665 704
2,World,900,Northern Africa and Western Asia,1833,1990,14 986 109,8 337 422,6 648 687
3,World,900,Central and Southern Asia,1831,1990,30 342 957,16 862 386,13 480 571
4,World,900,Eastern and South-Eastern Asia,1832,1990,14 465 509,7 069 837,7 395 672
...,...,...,...,...,...,...,...,...
224235,Wallis and Futuna Islands*,876,New Caledonia*,540,2024,1016,506,510
224236,Wallis and Futuna Islands*,876,Vanuatu,548,2024,67,39,28
224237,Wallis and Futuna Islands*,876,Polynesia*,957,2024,10,5,5
224238,Wallis and Futuna Islands*,876,French Polynesia*,258,2024,10,5,5


## 5. Initial Data Inspection

Before cleaning, the dataset structure is inspected to understand the available variables.

The checks below help identify:

- column names
- available country fields
- available years
- migrant stock value columns
- fields needed for filtering and transformation


In [23]:
undesa.columns.tolist()

['Destination',
 'Location code of destination',
 'Origin',
 'Location code of origin',
 'Year',
 'Total',
 'Male',
 'Female']

## 6. Filtering Cameroon as Origin Country

The analysis focuses on Cameroon as the country of origin.

This step keeps only records where the origin country is Cameroon. This ensures that the following analysis focuses only on Cameroonian international migration.


In [24]:
undesa = undesa[undesa["Origin"] == "Cameroon"].copy()
undesa

,Destination,Location code of destination,Origin,Location code of origin,Year,Total,Male,Female
49,World,900,Cameroon,120,1990,92 391,49 277,43 114
335,Sub-Saharan Africa,1834,Cameroon,120,1990,63 197,33 785,29 412
570,Northern Africa and Western Asia,1833,Cameroon,120,1990,65,51,14
765,Central and Southern Asia,1831,Cameroon,120,1990,..,..,..
914,Eastern and South-Eastern Asia,1832,Cameroon,120,1990,315,189,126
...,...,...,...,...,...,...,...,...
221814,NORTHERN AMERICA,905,Cameroon,120,2024,43 662,20 745,22 917
222076,Canada,124,Cameroon,120,2024,43 662,20 745,22 917
222421,OCEANIA,909,Cameroon,120,2024,695,397,298
222677,Australia/New Zealand,927,Cameroon,120,2024,695,397,298


## 7. Numeric Conversion of Migrant Stock Values

The columns `Total`, `Male` and `Female` represent migrant stock values.

They are converted into numeric format to make them usable for aggregation, comparison and visualization.

Any non-numeric value is converted to a missing value so it can be checked later.


In [25]:
for col in ["Total", "Male", "Female"]:
    undesa[col] = undesa[col].astype(str).str.replace(" ", "", regex=False)
    undesa[col] = pd.to_numeric(undesa[col], errors="coerce")
undesa

,Destination,Location code of destination,Origin,Location code of origin,Year,Total,Male,Female
49,World,900,Cameroon,120,1990,92391.0,49277.0,43114.0
335,Sub-Saharan Africa,1834,Cameroon,120,1990,63197.0,33785.0,29412.0
570,Northern Africa and Western Asia,1833,Cameroon,120,1990,65.0,51.0,14.0
765,Central and Southern Asia,1831,Cameroon,120,1990,NaN,NaN,NaN
914,Eastern and South-Eastern Asia,1832,Cameroon,120,1990,315.0,189.0,126.0
...,...,...,...,...,...,...,...,...
221814,NORTHERN AMERICA,905,Cameroon,120,2024,43662.0,20745.0,22917.0
222076,Canada,124,Cameroon,120,2024,43662.0,20745.0,22917.0
222421,OCEANIA,909,Cameroon,120,2024,695.0,397.0,298.0
222677,Australia/New Zealand,927,Cameroon,120,2024,695.0,397.0,298.0


## 8. Year Coverage Check

This step checks which reference years are available in the dataset.

The project focuses on the 2015–2024 period, but UN DESA migrant stock data is generally available by reference years rather than as continuous annual observations.


In [26]:
sorted(undesa["Year"].unique())

[np.int64(1990),
 np.int64(1995),
 np.int64(2000),
 np.int64(2005),
 np.int64(2010),
 np.int64(2015),
 np.int64(2020),
 np.int64(2024)]

## 9. Destination Coverage Check

The next checks inspect the number and labels of destination countries.

This helps detect whether the dataset contains only countries or also broader regional aggregates that should not be treated as countries.


In [27]:
undesa["Destination"].nunique()

91

In [28]:
undesa["Destination"].drop_duplicates().sort_values().tolist()[:50]

['AFRICA',
 'ASIA',
 'Australia*',
 'Australia/New Zealand',
 'Belgium',
 'Bolivia (Plurinational State of)',
 'Brazil',
 'Bulgaria',
 'Canada',
 'Caribbean',
 'Central America',
 'Central and Southern Asia',
 'Chad',
 'Chile',
 'China',
 'Congo',
 'Costa Rica',
 'Cyprus',
 'Denmark*',
 'Dominican Republic',
 'EUROPE',
 'Eastern Africa',
 'Eastern Asia',
 'Eastern Europe',
 'Eastern and South-Eastern Asia',
 'Egypt',
 'Equatorial Guinea',
 'Estonia',
 'Europe and Northern America',
 'Finland*',
 'France*',
 'Gabon',
 'Ghana',
 'Greece',
 'Guinea',
 'High-and-upper-middle-income countries',
 'High-income countries',
 'Hungary',
 'Iceland',
 'India',
 'Italy',
 'Japan',
 'LATIN AMERICA AND THE CARIBBEAN',
 'Land-locked Developing Countries (LLDC)',
 'Latin America and the Caribbean',
 'Latvia',
 'Least developed countries',
 'Less developed regions',
 'Less developed regions, excluding China',
 'Less developed regions, excluding least developed countries']

## 10. Removing Regional and Income-Level Aggregates

UN DESA data may include regional groups, income groups or global aggregates such as `World`, `Africa`, `Europe`, or `High-income countries`.

These rows are removed because the analysis focuses on destination countries, not regional or economic aggregates.

Keeping these aggregates would distort the ranking of destination countries and create double-counting risks.


In [29]:
aggregates_to_remove = [
    "World",
    "More developed regions",
    "Less developed regions",
    "Less developed regions, excluding least developed countries",
    "Less developed regions, excluding China",

    "AFRICA",
    "ASIA",
    "EUROPE",
    "OCEANIA",
    "NORTHERN AMERICA",
    "LATIN AMERICA AND THE CARIBBEAN",

    "Africa",
    "Asia",
    "Europe and Northern America",
    "Latin America and the Caribbean",

    "Sub-Saharan Africa",
    "Northern Africa and Western Asia",
    "Central and Southern Asia",
    "Eastern and South-Eastern Asia",

    "Eastern Africa",
    "Middle Africa",
    "Northern Africa",
    "Southern Africa",
    "Western Africa",

    "Caribbean",
    "Central America",
    "South America",

    "Central Asia",
    "Eastern Asia",
    "Southern Asia",
    "South-Eastern Asia",
    "Western Asia",

    "Eastern Europe",
    "Northern Europe",
    "Southern Europe",
    "Western Europe",

    "Australia/New Zealand",
    "Oceania (excluding Australia and New Zealand)",
    "Melanesia",
    "Micronesia",
    "Polynesia*",

    "Least developed countries",
    "Land-locked Developing Countries (LLDC)",
    "Small Island Developing States (SIDS)",

    "Low-income countries",
    "Lower-middle-income countries",
    "Upper-middle-income countries",
    "High-income countries",
    "Middle-income countries",
    "Low-and-middle-income countries",
    "Low-and-Lower-middle-income countries",
    "High-and-upper-middle-income countries",
    "No income group available",
]

undesa_clean = undesa[
    ~undesa["Destination"].isin(aggregates_to_remove)
].copy()

undesa_clean


,Destination,Location code of destination,Origin,Location code of origin,Year,Total,Male,Female
7130,Zambia,894,Cameroon,120,1990,10.0,8.0,2.0
7467,Chad,148,Cameroon,120,1990,30992.0,16731.0,14261.0
7508,Congo,178,Cameroon,120,1990,3464.0,2036.0,1428.0
7583,Equatorial Guinea,226,Cameroon,120,1990,202.0,133.0,69.0
7620,Gabon,266,Cameroon,120,1990,15661.0,7765.0,7896.0
...,...,...,...,...,...,...,...,...
220610,Bolivia (Plurinational State of),68,Cameroon,120,2024,15.0,14.0,1.0
220828,Brazil,76,Cameroon,120,2024,723.0,532.0,191.0
220984,Chile,152,Cameroon,120,2024,281.0,49.0,232.0
222076,Canada,124,Cameroon,120,2024,43662.0,20745.0,22917.0


## 11. Selecting Relevant Reference Years

This step keeps the reference years used in the project: 2015, 2020 and 2024.

These years allow the project to compare the situation before, during and after the Covid-19 period, while remaining aligned with the years available in the UN DESA dataset.


In [30]:
undesa_clean = undesa_clean[undesa_clean["Year"].isin([2015, 2020, 2024])].copy()
sorted(undesa_clean["Year"].unique())
undesa_clean

,Destination,Location code of destination,Origin,Location code of origin,Year,Total,Male,Female
147280,Zambia,894,Cameroon,120,2015,214.0,114.0,100.0
147617,Chad,148,Cameroon,120,2015,30702.0,13908.0,16794.0
147658,Congo,178,Cameroon,120,2015,11300.0,6727.0,4573.0
147733,Equatorial Guinea,226,Cameroon,120,2015,798.0,535.0,263.0
147770,Gabon,266,Cameroon,120,2015,46269.0,26905.0,19364.0
...,...,...,...,...,...,...,...,...
220610,Bolivia (Plurinational State of),68,Cameroon,120,2024,15.0,14.0,1.0
220828,Brazil,76,Cameroon,120,2024,723.0,532.0,191.0
220984,Chile,152,Cameroon,120,2024,281.0,49.0,232.0
222076,Canada,124,Cameroon,120,2024,43662.0,20745.0,22917.0


## 12. Column Standardization

Column names are renamed using clear and consistent labels.

This makes the dataset easier to read and prepares it for integration with the project-wide analytical schema.

Key standardized fields include:

- `destination_country`
- `destination_code`
- `origin_country`
- `origin_code`
- `year`
- `total_migrants`
- `male_migrants`
- `female_migrants`


In [31]:
undesa_clean = undesa_clean.rename(columns={
    "Destination": "destination_country",
    "Location code of destination": "destination_code",
    "Origin": "origin_country",
    "Location code of origin": "origin_code",
    "Year": "year",
    "Total": "total_migrants",
    "Male": "male_migrants",
    "Female": "female_migrants"
})
undesa_clean

,destination_country,destination_code,origin_country,origin_code,year,total_migrants,male_migrants,female_migrants
147280,Zambia,894,Cameroon,120,2015,214.0,114.0,100.0
147617,Chad,148,Cameroon,120,2015,30702.0,13908.0,16794.0
147658,Congo,178,Cameroon,120,2015,11300.0,6727.0,4573.0
147733,Equatorial Guinea,226,Cameroon,120,2015,798.0,535.0,263.0
147770,Gabon,266,Cameroon,120,2015,46269.0,26905.0,19364.0
...,...,...,...,...,...,...,...,...
220610,Bolivia (Plurinational State of),68,Cameroon,120,2024,15.0,14.0,1.0
220828,Brazil,76,Cameroon,120,2024,723.0,532.0,191.0
220984,Chile,152,Cameroon,120,2024,281.0,49.0,232.0
222076,Canada,124,Cameroon,120,2024,43662.0,20745.0,22917.0


## 13. Adding Source Metadata

Source metadata is added to make the cleaned dataset traceable.

The added fields specify:

- the data source: `UN DESA`
- the dataset type: `international_migrant_stock`
- the measurement type: `stock`

This is important because stock indicators must remain separated from flow indicators such as first permits or citizenship acquisitions.


In [32]:
undesa_clean["source"] = "UN DESA"
undesa_clean["dataset"] = "international_migrant_stock"
undesa_clean["measure_type"] = "stock"

undesa_clean

,destination_country,destination_code,origin_country,origin_code,year,total_migrants,male_migrants,female_migrants,source,dataset,measure_type
147280,Zambia,894,Cameroon,120,2015,214.0,114.0,100.0,UN DESA,international_migrant_stock,stock
147617,Chad,148,Cameroon,120,2015,30702.0,13908.0,16794.0,UN DESA,international_migrant_stock,stock
147658,Congo,178,Cameroon,120,2015,11300.0,6727.0,4573.0,UN DESA,international_migrant_stock,stock
147733,Equatorial Guinea,226,Cameroon,120,2015,798.0,535.0,263.0,UN DESA,international_migrant_stock,stock
147770,Gabon,266,Cameroon,120,2015,46269.0,26905.0,19364.0,UN DESA,international_migrant_stock,stock
...,...,...,...,...,...,...,...,...,...,...,...
220610,Bolivia (Plurinational State of),68,Cameroon,120,2024,15.0,14.0,1.0,UN DESA,international_migrant_stock,stock
220828,Brazil,76,Cameroon,120,2024,723.0,532.0,191.0,UN DESA,international_migrant_stock,stock
220984,Chile,152,Cameroon,120,2024,281.0,49.0,232.0,UN DESA,international_migrant_stock,stock
222076,Canada,124,Cameroon,120,2024,43662.0,20745.0,22917.0,UN DESA,international_migrant_stock,stock


## 14. Quality Checks After Cleaning

After cleaning, quality checks are performed to verify the reliability of the output.

The main checks include:

- data types
- first rows of the cleaned dataset
- missing values in each column
- consistency of standardized fields

These checks help ensure that the cleaned dataset is ready for the next steps of the pipeline.


In [33]:
undesa_clean.info()
undesa_clean.head()
undesa_clean.isna().sum()

<class 'pandas.DataFrame'>
Index: 141 entries, 147280 to 222928
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   destination_country  141 non-null    str    
 1   destination_code     141 non-null    int64  
 2   origin_country       141 non-null    str    
 3   origin_code          141 non-null    int64  
 4   year                 141 non-null    int64  
 5   total_migrants       141 non-null    float64
 6   male_migrants        141 non-null    float64
 7   female_migrants      138 non-null    float64
 8   source               141 non-null    str    
 9   dataset              141 non-null    str    
 10  measure_type         141 non-null    str    
dtypes: float64(3), int64(3), str(5)
memory usage: 13.2 KB


destination_country    0
destination_code       0
origin_country         0
origin_code            0
year                   0
total_migrants         0
male_migrants          0
female_migrants        3
source                 0
dataset                0
measure_type           0
dtype: int64

## 15. Output Export

The cleaned UN DESA dataset is exported to the processed data folder.

Output file:

```text
data/processed/cleaned/undesa_destinations_cleaned.csv
```

This cleaned file will later be used to build the final destination-level analytical table.


In [34]:
undesa_clean.to_csv(
    CLEANED_PATH / "undesa_destinations_cleaned.csv",
    index=False
)
undesa_clean = undesa_clean.reset_index(drop=True)
undesa_clean

,destination_country,destination_code,origin_country,origin_code,year,total_migrants,male_migrants,female_migrants,source,dataset,measure_type
0,Zambia,894,Cameroon,120,2015,214.0,114.0,100.0,UN DESA,international_migrant_stock,stock
1,Chad,148,Cameroon,120,2015,30702.0,13908.0,16794.0,UN DESA,international_migrant_stock,stock
2,Congo,178,Cameroon,120,2015,11300.0,6727.0,4573.0,UN DESA,international_migrant_stock,stock
3,Equatorial Guinea,226,Cameroon,120,2015,798.0,535.0,263.0,UN DESA,international_migrant_stock,stock
4,Gabon,266,Cameroon,120,2015,46269.0,26905.0,19364.0,UN DESA,international_migrant_stock,stock
...,...,...,...,...,...,...,...,...,...,...,...
136,Bolivia (Plurinational State of),68,Cameroon,120,2024,15.0,14.0,1.0,UN DESA,international_migrant_stock,stock
137,Brazil,76,Cameroon,120,2024,723.0,532.0,191.0,UN DESA,international_migrant_stock,stock
138,Chile,152,Cameroon,120,2024,281.0,49.0,232.0,UN DESA,international_migrant_stock,stock
139,Canada,124,Cameroon,120,2024,43662.0,20745.0,22917.0,UN DESA,international_migrant_stock,stock


## 16. Limitations

This dataset should be interpreted with caution.

Main limitations:

- it measures migrant stock, not new migration flows
- it is based on available reference years rather than continuous annual observations
- it does not explain individual migration motivations
- it does not track individual migrant journeys
- differences between countries may reflect both migration dynamics and statistical reporting methods

As a result, the dataset is useful for identifying broad destination patterns, but it should not be used to make unsupported claims about individual behavior or direct Covid-19 causality.


## 17. Conclusion

This notebook produces a cleaned version of the UN DESA migrant stock dataset.

The cleaned output is ready to be used in the standardization step and later in the construction of the `destinations_master.csv` analytical table.

This source contributes to the analysis of the main destination countries of Cameroonian migrants and their evolution across available reference years.
